# Crypto OHLCV Downloader - 1w
Downloads **1w** klines via Binance REST API / CCXT (or resamples directly from 1d klines), and saves one Parquet file per symbol in Google Drive.

Run the cells top to bottom. Re-running is incremental and covers the complete history from 2020 through today.

## 1. Install & mount Drive

In [ ]:
!pip install -q ccxt pyarrow python-dateutil requests pandas

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Running in local environment (Google Colab drive mount skipped).")

## 2. Config

In [ ]:
# ============ CONFIG (edit here) ============
TF          = "1w"                 # timeframe for this notebook
SYMBOLS     = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT"]
START_DATE  = "2020-01-01"             # earliest date to download (YYYY-MM-DD)
END_DATE    = None                     # latest date to download (None = today e.g. 2026-10-25)
MARKET      = "futures/um"             # USDT-M perpetual (per spec); use "spot" for spot
SAVE_CSV    = False                    # parquet is always saved; set True to also save CSV
DEFAULT_DIR = "/content/drive/MyDrive/crypto_data" if ("IN_COLAB" in globals() and IN_COLAB) else r"D:\Projects_Main\Trade-Pro\Trader-Pro\notebooks\data"
# ============================================

## 3. Helpers

In [ ]:
import os, io, zipfile, time
import requests
import pandas as pd
from datetime import datetime, timezone, timedelta
from dateutil.relativedelta import relativedelta

OUT_DIR = os.path.join(DRIVE_DIR, TF)
os.makedirs(OUT_DIR, exist_ok=True)

COLS = ["open_time","open","high","low","close","volume","close_time",
        "quote_volume","trades","taker_buy_base","taker_buy_quote","ignore"]
KEEP = ["open_time","open","high","low","close","volume","close_time",
        "quote_volume","trades","taker_buy_base","taker_buy_quote"]
TF_DELTA = pd.Timedelta(days=7)
print("Output folder:", OUT_DIR)

In [ ]:
def normalize(df):
    """Clean a raw Binance kline dataframe -> typed, UTC-indexed-ready frame."""
    df = df.copy()
    df = df[pd.to_numeric(df["open_time"], errors="coerce").notna()]   # drop header rows
    for c in COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    # newer files can be in microseconds; convert to milliseconds
    for c in ["open_time", "close_time"]:
        if c in df.columns:
            df.loc[df[c] > 1e14, c] = df.loc[df[c] > 1e14, c] // 1000
    df["open_time"]  = pd.to_datetime(df["open_time"].astype("int64"),  unit="ms", utc=True)
    df["close_time"] = pd.to_datetime(df["close_time"].astype("int64"), unit="ms", utc=True)
    if "trades" in df.columns:
        df["trades"] = df["trades"].fillna(0).astype("int64")
    for c in KEEP:
        if c not in df.columns:
            df[c] = float("nan")
    return df[KEEP]


def resample_1d_to_1w(df_1d):
    """Resample daily klines to Binance weekly candles (Monday 00:00 to Sunday 23:59:59)."""
    df = df_1d.copy().sort_values("open_time").set_index("open_time")
    w = df.resample("W-MON", closed="left", label="left").agg({
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "volume": "sum",
        "quote_volume": "sum",
        "trades": "sum",
        "taker_buy_base": "sum",
        "taker_buy_quote": "sum",
        "close_time": "last"
    }).dropna(subset=["open", "close"]).reset_index()
    return w[KEEP]


def fetch_binance_rest_1w(symbol, start_dt, market):
    """Fetch weekly candles directly from Binance REST API (one call covers entire history)."""
    endpoint = "https://fapi.binance.com/fapi/v1/klines" if market == "futures/um" else "https://api.binance.com/api/v3/klines"
    start_ms = int(start_dt.timestamp() * 1000)
    r = requests.get(endpoint, params={"symbol": symbol, "interval": "1w", "startTime": start_ms, "limit": 1000}, timeout=15)
    if r.status_code == 200 and isinstance(r.json(), list) and len(r.json()) > 0:
        raw = pd.DataFrame(r.json(), columns=COLS[:len(r.json()[0])])
        return normalize(raw)
    return None

## 4. Download & save

In [ ]:
def process_symbol(symbol):
    path = os.path.join(OUT_DIR, f"{symbol}_{TF}.parquet")
    existing = pd.read_parquet(path) if os.path.exists(path) else None

    start_dt = datetime.fromisoformat(START_DATE).replace(tzinfo=timezone.utc)
    if END_DATE is not None:
        end_dt = datetime.fromisoformat(END_DATE).replace(tzinfo=timezone.utc)
    else:
        end_dt = datetime.now(timezone.utc)

    print(f"[{symbol}] fetching 1w klines ...")
    df = None

    # Method 1: Direct Binance REST endpoint (fetches all weekly bars in 1 call)
    try:
        df = fetch_binance_rest_1w(symbol, start_dt, MARKET)
        if df is not None:
            print(f"[{symbol}] pulled {len(df)} 1w candles via Binance REST")
    except Exception as e:
        print(f"[{symbol}] REST fetch skipped ({e})")

    # Method 2: Resample from 1d parquet file if available
    if df is None or len(df) == 0:
        p_1d = os.path.join(DRIVE_DIR, "1d", f"{symbol}_1d.parquet")
        if os.path.exists(p_1d):
            print(f"[{symbol}] resampling from 1d parquet file: {p_1d}")
            d1 = pd.read_parquet(p_1d)
            df = resample_1d_to_1w(d1)
        else:
            print(f"[{symbol}] attempting fallback via CCXT ...")
            try:
                import ccxt
                ex = ccxt.binanceusdm({"enableRateLimit": True}) if MARKET == "futures/um" else ccxt.binance({"enableRateLimit": True})
                sym = symbol.replace("USDT", "/USDT:USDT") if MARKET == "futures/um" else symbol.replace("USDT", "/USDT")
                batch = ex.fetch_ohlcv(sym, "1w", since=int(start_dt.timestamp() * 1000), limit=1000)
                if batch:
                    rows = []
                    dur = 7 * 24 * 60 * 60 * 1000
                    for b in batch:
                        ct = b[0] + dur - 1
                        qv = b[5] * (b[1] + b[2] + b[3] + b[4]) / 4
                        rows.append([b[0], b[1], b[2], b[3], b[4], b[5], ct, qv, 0, float("nan"), float("nan"), 0])
                    raw = pd.DataFrame(rows, columns=COLS[:len(rows[0])])
                    df = normalize(raw)
            except Exception as e:
                print(f"[{symbol}] CCXT failed: {e}")

    if existing is not None and df is not None:
        df = pd.concat([existing, df], ignore_index=True)

    if df is None or len(df) == 0:
        print(f"[{symbol}] no data could be retrieved.")
        return None

    df = df.drop_duplicates("open_time").sort_values("open_time").reset_index(drop=True)

    # Filter to END_DATE if explicitly specified
    if END_DATE is not None:
        cutoff = pd.to_datetime(END_DATE, utc=True) + pd.Timedelta(days=7)
        df = df[df["open_time"] < cutoff].reset_index(drop=True)

    df.to_parquet(path, index=False)
    if SAVE_CSV:
        df.to_csv(path.replace(".parquet", ".csv"), index=False)
    print(f"[{symbol}] saved {len(df):,} weekly bars ({df['open_time'].iloc[0]} -> {df['open_time'].iloc[-1]})")
    return df


results = {}
for s in SYMBOLS:
    results[s] = process_symbol(s)
print("\nDone.")

## 5. Quality check

In [ ]:
# ---- Data quality report ----
rows = []
for s, df in results.items():
    if df is None or df.empty:
        rows.append([s, 0, None, None, None, None]); continue
    diffs = df["open_time"].diff().dropna()
    n_gaps = int((diffs != TF_DELTA).sum())
    max_gap = diffs.max() if len(diffs) else None
    rows.append([s, len(df), df["open_time"].iloc[0], df["open_time"].iloc[-1], n_gaps, max_gap])
report = pd.DataFrame(rows, columns=["symbol","bars","first_bar","last_bar","irregular_steps","largest_gap"])
display(report)

# peek at one file
first = next((d for d in results.values() if d is not None), None)
if first is not None:
    display(first.tail())